In [ ]:
from google.colab import drive
mounted=drive.mount('content')

Drive already mounted at content; to attempt to forcibly remount, call drive.mount("content", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, RobertaModel, get_scheduler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
df=pd.read_csv('/content/content/MyDrive/Colab Notebooks/fake_job_postings.csv')

In [ ]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry             12977 non-null  object
 16  func

In [ ]:
df=df.drop('job_id',axis=1)
df=df.drop('department',axis=1)
df=df.drop('salary_range',axis=1)
df=df.drop('telecommuting',axis=1)
df=df.drop('benefits',axis=1)
df=df.drop('has_company_logo',axis=1)
df=df.drop('has_questions',axis=1)

In [ ]:
df=df.drop_duplicates()

In [ ]:
round((df.isnull().sum()/17880)*100,3)

,0
title,0.000
location,1.907
company_profile,18.333
description,0.006
requirements,14.793
employment_type,19.150
required_experience,38.915
required_education,44.771
industry,27.030
function,35.548


In [ ]:
df['location'] = df['location'].fillna(df['location'].mode()[0])
df['description'] = df['description'].fillna(df['description'].mode()[0])

In [ ]:
cols = ['company_profile','requirements','employment_type','required_experience','required_education', 'industry','function']

for col in cols:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

df['employment_type'] = df['employment_type'].fillna('Not Specified')
df['required_experience'] = df['required_experience'].fillna('Not Specified')
df['required_education'] = df['required_education'].fillna('Not Specified')
df['function'] = df['function'].fillna('Not Specified')
df['industry'] = df['industry'].fillna('Unknown')

In [ ]:
x=df.drop('fraudulent',axis=1)
y=df['fraudulent']

In [ ]:
cat_cols=['employment_type','required_experience','required_education','function']
num_cols=x.select_dtypes(include=['int64','float64']).columns.tolist()
text_cols=['title','location','company_profile','description','requirements','industry']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=13)

In [ ]:
def make_combined_text(df_subset):
    return (
        df_subset['title'].fillna('')            + ' [SEP] ' +
        df_subset['description'].fillna('')      + ' [SEP] ' +
        df_subset['requirements'].fillna('')     + ' [SEP] ' +
        df_subset['company_profile'].fillna('')  + ' [SEP] ' +
        df_subset['location'].fillna('')         + ' [SEP] ' +
        df_subset['industry'].fillna('')
    )

x_train['combined_text'] = make_combined_text(x_train)
x_test['combined_text']  = make_combined_text(x_test)

x_train=x_train.drop(columns=text_cols)
x_test=x_test.drop(columns=text_cols)

In [ ]:
for col in cat_cols:
    le = LabelEncoder()
    x_train[col] = le.fit_transform(x_train[col])
    x_test[col] = le.transform(x_test[col])  # only transform, never fit

In [ ]:
scaler = StandardScaler()

x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])

In [ ]:
class CustomDataset(Dataset):

    def __init__(self,text_features,num_features,cat_features,labels):
        self.text_features = text_features.tolist()
        self.num_features = torch.tensor(num_features.values,dtype=torch.float32)
        self.cat_features = torch.tensor(cat_features.values,dtype=torch.long)
        self.labels = torch.tensor(labels.values,dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.text_features[idx],self.num_features[idx],self.cat_features[idx],self.labels[idx])

In [ ]:
train_dataset = CustomDataset(
    x_train['combined_text'],
    x_train[num_cols],
    x_train[cat_cols],
    y_train
)

test_dataset = CustomDataset(
    x_test['combined_text'],
    x_test[num_cols],
    x_test[cat_cols],
    y_test
)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("distilroberta-base")

def collate_fn(batch):

  texts,nums,cats,labels=zip(*batch)

  encoding=tokenizer(list(texts),padding=True,truncation=True,max_length=512,return_tensors='pt')

  return{
      'input_ids':encoding['input_ids'],
      'attention_mask':encoding['attention_mask'],
      'num_features': torch.stack(nums),
      'cat_features': torch.stack(cats),
      'label': torch.stack(labels)
  }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    collate_fn=collate_fn
)

In [ ]:
class NeuralNetwork(nn.Module):

  def __init__(self,num_features_count,cat_col_dim,embedding_dim):
    super().__init__()

    self.model = RobertaModel.from_pretrained("distilroberta-base")

    for param in self.model.embeddings.parameters():
          param.requires_grad = False

    for layer_idx in range(6):
          for param in self.model.encoder.layer[layer_idx].parameters():
              param.requires_grad = False

    roberta_output_dim=768

    self.text_projector=nn.Sequential(
        nn.Linear(roberta_output_dim,384),
        nn.ReLU(),
        nn.Dropout(0.3)
    )

    self.num_class=nn.Sequential(
        nn.Linear(num_features_count,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(64,32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.1)
    )

    embeddings = []

    for num_categories in cat_col_dim:
      embeddings.append(
          nn.Embedding(num_categories + 1, embedding_dim)
      )

    self.embeddings = nn.ModuleList(embeddings)

    cat_output_dim = len(cat_col_dim) * embedding_dim

    self.cat_class=nn.Sequential(
        nn.Linear(cat_output_dim,64),
        nn.ReLU(),
        nn.Dropout(0.3)
    )

    fusion_input_dim = 384 + 32 + 64

    self.fusion_head = nn.Sequential(
          nn.Linear(fusion_input_dim, 128),
          nn.BatchNorm1d(128),
          nn.ReLU(),
          nn.Dropout(0.3),

          nn.Linear(128, 64),
          nn.ReLU(),
          nn.Dropout(0.3),

          nn.Linear(64, 1)
      )


  def forward(self,input_ids,attention_mask,num_features,cat_features):

    roberta_output=self.model(input_ids=input_ids,attention_mask=attention_mask)

    cls_token = roberta_output.last_hidden_state[:, 0, :]

    text_features = self.text_projector(cls_token)

    numerical_features = self.num_class(num_features)

    embedded_features=[]

    for i in range(cat_features.shape[1]):
      single_col=cat_features[:,i]
      embedded_col=self.embeddings[i](single_col)
      embedded_features.append(embedded_col)

    # Combine all categorical embeddings
    categorical_features = torch.cat(embedded_features, dim=1)

    # Pass through categorical branch
    categorical_features = self.cat_class(categorical_features)

    combined_features = torch.cat(
          [text_features, numerical_features, categorical_features], dim=1)

    output = self.fusion_head(combined_features)

    output = output.squeeze(1)

    return output

In [ ]:
class FocalLoss(nn.Module):

    def __init__(self, alpha, gamma):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce)

        # alpha for positive class (fraud), (1-alpha) for negative class
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        focal_weight = (1 - pt) ** self.gamma
        loss = alpha_t * focal_weight * bce
        return loss.mean()

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience   = patience
        self.best_score = 0
        self.counter    = 0

    def step(self, score):
        if score > self.best_score:
            self.best_score = score
            self.counter    = 0
            return False        # don't stop
        self.counter += 1
        print(f"  EarlyStopping counter: {self.counter}/{self.patience}")
        return self.counter >= self.patience  # stop=True

In [ ]:
num_features_count = len(num_cols)
cat_col_dim = [
    x_train[col].nunique()
    for col in cat_cols
]

In [ ]:
fraud_count = y_train.sum()
total = len(y_train)
alpha = 1 - (fraud_count / total)

In [ ]:
model = NeuralNetwork(
    num_features_count = num_features_count,
    cat_col_dim = cat_col_dim,
    embedding_dim = 9
    ).to(device)

criterion = FocalLoss(
    alpha = alpha,
    gamma = 2.0
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
optimizer = AdamW([
    {"params": model.model.parameters(),        "lr": 2e-5},
    {"params": model.text_projector.parameters(), "lr": 1e-4},
    {"params": model.num_class.parameters(),     "lr": 1e-4},
    {"params": model.embeddings.parameters(),     "lr": 1e-4},
    {"params": model.cat_class.parameters(),     "lr": 1e-4},
    {"params": model.fusion_head.parameters(),    "lr": 1e-4},
], weight_decay=0.01)

# scheduler
num_epochs   = 5
total_steps  = num_epochs * len(train_loader)
warmup_steps = int(0.1 * total_steps)

scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = warmup_steps,
    num_training_steps = total_steps
)

In [ ]:
def train_model(model, train_loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        num_features   = batch["num_features"].to(device)
        cat_features   = batch["cat_features"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_features=num_features,
            cat_features=cat_features
        )

        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
def find_best_threshold(all_probs, all_labels):
    """Find threshold that maximizes F1"""
    best_thresh, best_f1 = 0.5, 0
    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = (np.array(all_probs) >= thresh).astype(float)
        f1    = f1_score(all_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1    = f1
            best_thresh= thresh
    return best_thresh, best_f1

In [ ]:
def evaluate_model(model, test_loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            num_features   = batch["num_features"].to(device)
            cat_features   = batch["cat_features"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                num_features=num_features,
                cat_features=cat_features
            )

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Find best threshold
    best_thresh, _ = find_best_threshold(all_probs, all_labels)
    preds = (np.array(all_probs) >= best_thresh).astype(float)

    accuracy  = accuracy_score(all_labels, preds)
    precision = precision_score(all_labels, preds, zero_division=0)
    recall    = recall_score(all_labels, preds, zero_division=0)
    f1        = f1_score(all_labels, preds, zero_division=0)

    return total_loss / len(test_loader), accuracy, precision, recall, f1, best_thresh

In [ ]:
early_stopping = EarlyStopping(patience=3)

best_f1 = 0

for epoch in range(num_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch [{epoch+1}/{num_epochs}]")

    train_loss = train_model(
        model, train_loader, criterion, optimizer, scheduler, device
    )

    test_loss, acc, prec, rec, f1, best_thresh = evaluate_model(
        model, test_loader, criterion, device
    )

    print(
        f"Train Loss : {train_loss:.4f}\n"
        f"Test Loss  : {test_loss:.4f}\n"
        f"Accuracy   : {acc:.4f}\n"
        f"Precision  : {prec:.4f}\n"
        f"Recall     : {rec:.4f}\n"
        f"F1 Score   : {f1:.4f}\n"
        f"Best Thresh: {best_thresh:.2f}"
    )

    # Save best model based on F1
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"Best model saved! (F1={f1:.4f})")

    # Early stopping check
    if early_stopping.step(f1):
        print("Early stopping triggered!")
        break

print(f"\nTraining complete! Best F1: {best_f1:.4f}")

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
print("Best model loaded for inference")


Epoch [1/5]
Train Loss : 0.0160
Test Loss  : 0.0142
Accuracy   : 0.9217
Precision  : 0.2411
Recall     : 0.2864
F1 Score   : 0.2618
Best Thresh: 0.55
Best model saved! (F1=0.2618)

Epoch [2/5]
Train Loss : 0.0141
Test Loss  : 0.0126
Accuracy   : 0.8962
Precision  : 0.2453
Recall     : 0.5493
F1 Score   : 0.3391
Best Thresh: 0.55
Best model saved! (F1=0.3391)

Epoch [3/5]
Train Loss : 0.0129
Test Loss  : 0.0119
Accuracy   : 0.9132
Precision  : 0.2789
Recall     : 0.4977
F1 Score   : 0.3575
Best Thresh: 0.60
Best model saved! (F1=0.3575)

Epoch [4/5]
Train Loss : 0.0123
Test Loss  : 0.0114
Accuracy   : 0.9187
Precision  : 0.3054
Recall     : 0.5305
F1 Score   : 0.3877
Best Thresh: 0.60
Best model saved! (F1=0.3877)

Epoch [5/5]
Train Loss : 0.0120
Test Loss  : 0.0112
Accuracy   : 0.9166
Precision  : 0.3013
Recall     : 0.5446
F1 Score   : 0.3880
Best Thresh: 0.60
Best model saved! (F1=0.3880)

Training complete! Best F1: 0.3880
Best model loaded for inference
